Interactive optimisation

In [ ]:
import asyncio
import numpy as np
import nglview as nv
from IPython.display import display

from ase import Atoms
from ase.calculators.emt import EMT
from ase.optimize import BFGS

molecules = [Atoms("H2O", positions=[(x, 0, 0), (x+1, 0, 0), (x+0.5, 1, 0)]) for x in [0,3,6]]
atoms = molecules[0] + molecules[1] + molecules[2]
# atoms.center(vacuum=4.0)
atoms.calc = EMT()

# Two-frame live trajectory: both frames reference the same mutable ASE Atoms.
live_frames = [atoms, atoms]
view = nv.show_asetraj(live_frames)
display(view)
await asyncio.sleep(0.5)
view.add_ball_and_stick()
view.center()

frame_counter = 0

def refresh_view():
    global frame_counter
    frame_counter = 1 - frame_counter
    # Toggle frame to force coordinate pull from the live ASE object.
    view.frame = frame_counter

opt = BFGS(atoms, logfile='log.txt')
refresh_view()                         # show initial geometry
for _ in range(500):
    converged = opt.run(fmax=0.05, steps=1)
    refresh_view()
    await asyncio.sleep(0.05)
    if converged:
        break
refresh_view()                         # show final geometry

NGLWidget(max_frame=1)